# 01_raw-to-aep

**Purpose**: Process raw BIDS EEG recordings into per-subject auditory evoked potentials (AEPs). For each intensity condition and each reference electrode, the pipeline filters, epochs, rejects artifacts, computes per-condition evoked responses, and saves results to BIDS derivatives.

**Workflow**:
1. Set tasks and parameters in the configuration cells (Sections 1 and 1b)
2. Resolve the subject list from the BIDS directory
3. For each intensity condition and reference electrode, re-analyze all subjects from raw data
4. Display and save per-subject trial-count summaries (heatmap + TSV)
5. Plot and save per-subject ERP comparison and topomap grid figures


## 0. Imports


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import mne

from erp_tools import (
    load_preprocess_epoch_multi_run, reject_epochs,
    compute_condition_evokeds,
    get_derivatives_root,
    save_subject_figure,
    save_epochs, save_condition_evokeds,
    save_subject_trial_counts,
    plot_evoked_comparison,
)
from erp_tools.profiles import marmoset_surface_eeg, human_eeg_32ch

mne.set_log_level('WARNING')


## 1. Configuration (edit this cell)


In [ ]:
# --- BIDS dataset ---
BIDS_ROOT = '/workspace/data/marmoset_FreqIntensity'  # TODO: update to actual dataset path
SESSION = None
RUNS = ['1', '2', '3', '4']  # None = single unspecified / ['1'] = single explicit / ['1','2'] = merge multiple

# All three intensity conditions to process
TASKS = {
    'Passive45dB': ['45dB_00125Hz', '45dB_00250Hz', '45dB_00500Hz', '45dB_01000Hz', '45dB_02000Hz', '45dB_04000Hz', '45dB_08000Hz', '45dB_16000Hz'],
    'Passive60dB': ['60dB_00125Hz', '60dB_00250Hz', '60dB_00500Hz', '60dB_01000Hz', '60dB_02000Hz', '60dB_04000Hz', '60dB_08000Hz', '60dB_16000Hz'],
    'Passive75dB': ['75dB_00125Hz', '75dB_00250Hz', '75dB_00500Hz', '75dB_01000Hz', '75dB_02000Hz', '75dB_04000Hz', '75dB_08000Hz', '75dB_16000Hz'],
}

# --- Subject filtering ---
# None: auto-detect all sub-XXX directories under BIDS_ROOT
# list: explicit list, e.g. ['001', '002']
SUBJECTS = None
EXCLUDE = []

# --- Grand average ---
COMPUTE_GROUP_AVERAGE = False
SHOW_CI = False
CI_LEVEL = 0.95

# --- Figure saving ---
SAVE_PER_SUBJECT_FIGURES = True
SAVE_GROUP_FIGURES = True

# --- Plot parameters ---
YLIM = (-6.0, 6.0)
EXCLUDE_FROM_TOPO = None
PICKS_FOR_COMPARE = None


## 1b. Analysis parameters

Specifies the species profile, filter band, epoch window, and artifact-rejection settings applied when running the pipeline from raw data. The `REFERENCES` list controls which reference electrodes are processed; the entire pipeline (preprocessing → evoked → figures) runs once per entry.


In [ ]:
# Select species profile (must match notebook 02)
filter_default, epoch_default, meta = marmoset_surface_eeg()
# filter_default, epoch_default, meta = human_eeg_32ch()

# --- Reference electrodes (both are processed in the loop below) ---
REFERENCES = [
    (['A1', 'A2'], 'linkedEars'),  # linked mastoids
    ('average',    'CAR'),         # common average reference (MNE built-in)
    ('median',     'CMR'),         # common median reference
]

MONTAGE = meta['montage']
EVENT_ID = None

# Epoch window
epoch_default.tmin = -0.1
epoch_default.tmax = 0.65

# Bandpass filter
filter_default.l_freq = 1.0
filter_default.h_freq = 100.0
filter_default.resample_sfreq = 1000.0

# Epoch rejection
REJECTION_METHOD = 'sd'
SD_N = 2.0
SD_METHOD = 'mad'

print(f'Re-analysis settings:')
print(f'  species   : {meta["species"]}')
print(f'  tmin/tmax : {epoch_default.tmin} / {epoch_default.tmax}')
print(f'  filter    : {filter_default.l_freq} - {filter_default.h_freq} Hz')
print(f'  rejection : {REJECTION_METHOD} (n_sd={SD_N}, method={SD_METHOD})')
print(f'  references: {[r for _, r in REFERENCES]}')


## 2. Resolve subject list


In [ ]:
bids_root_path = Path(BIDS_ROOT)

# Resolve subject list (independent of reference electrode choice)
if SUBJECTS is None:
    found = sorted([
        p.name.replace('sub-', '')
        for p in bids_root_path.glob('sub-*')
        if p.is_dir()
    ])
    SUBJECTS_RESOLVED = [s for s in found if s not in EXCLUDE]
    print(f'Subjects found in BIDS ({len(SUBJECTS_RESOLVED)}): {SUBJECTS_RESOLVED}')
    if EXCLUDE:
        print(f'  Excluded: {EXCLUDE}')
else:
    SUBJECTS_RESOLVED = list(SUBJECTS)
    print(f'Explicitly specified subjects ({len(SUBJECTS_RESOLVED)}): {SUBJECTS_RESOLVED}')

if not SUBJECTS_RESOLVED:
    raise RuntimeError('No subjects to process. Check SUBJECTS / EXCLUDE / BIDS_ROOT.')


## 3. Per-subject processing loop

Defines the `reanalyze_subject` helper and runs the main processing loop. Iterates over each task in `TASKS` and each reference electrode in `REFERENCES`. For every combination, re-analyzes all resolved subjects from raw data, writes epochs and evoked files to BIDS derivatives, and generates trial-count summaries and per-subject figures.


In [ ]:
def reanalyze_subject(subject):
    """Re-analyze one subject from raw data (multi-run). Returns (epochs_clean, evokeds)."""
    raws, epochs = load_preprocess_epoch_multi_run(
        bids_root=BIDS_ROOT, subject=subject,
        task=TASK, runs=RUNS, session=SESSION,
        filter_default=filter_default, ref_channels=ref_channels, montage=MONTAGE,
        epoch_default=epoch_default, event_id=EVENT_ID,
    )
    result = reject_epochs(
        epochs,
        method=REJECTION_METHOD,
        sd_n=SD_N,
        sd_method=SD_METHOD,
        threshold_reject=epoch_default.reject,
        autoreject_n_interpolate=meta.get('autoreject_n_interpolate'),
    )
    evokeds = compute_condition_evokeds(result.epochs_clean)
    return result.epochs_clean, evokeds


### Main loop

Outer loop iterates over all tasks in `TASKS` (one per intensity level). Inner loop iterates over each reference electrode in `REFERENCES`. For every task/reference combination, re-analyzes all resolved subjects from raw data, saves results to derivatives, prints a trial-count summary, and generates per-subject figures.


In [ ]:
for TASK, EXPECTED_CONDITIONS in TASKS.items():
    print(f'\n{"#"*60}')
    print(f'Task: {TASK}')
    print(f'{"#"*60}')

    for ref_channels, ref_label in REFERENCES:
        PIPELINE_NAME = f'erp-pipeline-ref-{ref_label}'
        deriv_root    = get_derivatives_root(BIDS_ROOT, pipeline_name=PIPELINE_NAME)

        print(f'\n{"="*60}')
        print(f'Reference: {ref_label}  pipeline: {PIPELINE_NAME}')
        print(f'{"="*60}')

        evokeds_by_subject = {}
        epochs_by_subject = {}
        status_log = {}

        for sub in SUBJECTS_RESOLVED:
            try:
                print(f'→ analyzing sub-{sub}...')
                epochs_clean, evokeds = reanalyze_subject(sub)
                evokeds_by_subject[sub] = evokeds
                epochs_by_subject[sub] = epochs_clean
                status_log[sub] = 'reanalyzed'
            except Exception as e:
                print(f'✗ sub-{sub}: failed ({e})')
                status_log[sub] = 'failed'

        # Processing summary
        print('\n=== Processing summary ===')
        from collections import Counter
        counts_by_status = Counter(status_log.values())
        for status, n in counts_by_status.items():
            subs = [s for s, v in status_log.items() if v == status]
            print(f'  {status}: {n} subject(s) {subs}')

        if not evokeds_by_subject:
            raise RuntimeError('No subjects could be loaded.')

        # Save results to derivatives
        print('\n=== Saving results to derivatives ===')
        print(f'  pipeline: {PIPELINE_NAME}')
        for sub, epochs_clean in epochs_by_subject.items():
            evokeds = evokeds_by_subject[sub]
            save_kwargs = dict(
                bids_root=BIDS_ROOT, subject=sub, task=TASK, run=None,
                pipeline_name=PIPELINE_NAME,
            )
            save_epochs(epochs_clean, desc='clean', **save_kwargs)
            save_condition_evokeds(evokeds, **save_kwargs)
            print(f'  sub-{sub}: saved')

        # ── Section 4: Trial count summary ────────────────────────────────────
        all_conditions = set()
        for evs in evokeds_by_subject.values():
            all_conditions.update(evs.keys())
        all_conditions = sorted(all_conditions)

        counts = pd.DataFrame(
            index=sorted(evokeds_by_subject.keys()),
            columns=all_conditions,
            dtype='Int64',
        )
        for sub, evs in evokeds_by_subject.items():
            for cond, ev in evs.items():
                counts.loc[sub, cond] = ev.nave

        print('=== Trial count summary ===')
        print(counts)
        print(f'\nTotal trials per condition:')
        print(counts.sum())

        fig_counts, ax = plt.subplots(
            figsize=(max(4, 0.5 * len(all_conditions) + 2), max(3, 0.3 * len(counts) + 1.5)),
        )
        counts_num = counts.astype(float).fillna(0)
        im = ax.imshow(counts_num.values, aspect='auto', cmap='viridis')
        ax.set_xticks(range(len(all_conditions)))
        ax.set_xticklabels(all_conditions, rotation=45, ha='right')
        ax.set_yticks(range(len(counts)))
        ax.set_yticklabels([f'sub-{s}' for s in counts.index])
        ax.set_xlabel('Condition')
        ax.set_title(f'Trial counts (task-{TASK}, ref-{ref_label})')
        fig_counts.colorbar(im, ax=ax, label='# trials')

        for i in range(counts_num.shape[0]):
            for j in range(counts_num.shape[1]):
                val = counts_num.values[i, j]
                ax.text(j, i, f'{int(val)}', ha='center', va='center',
                        color='white' if val < counts_num.values.max() / 2 else 'black',
                        fontsize=8)

        fig_counts.tight_layout()
        plt.show()

        from IPython.display import display as _display
        _display(counts)

        # Save per-subject trial counts as individual TSV files under sub-XXX/eeg/
        if SAVE_PER_SUBJECT_FIGURES:
            for sub in counts.index:
                p = save_subject_trial_counts(
                    counts.loc[sub].dropna().astype(int),
                    bids_root=BIDS_ROOT, subject=sub, task=TASK,
                    pipeline_name=PIPELINE_NAME,
                )
                print(f'trial-count TSV sub-{sub}: {p}')

        # ── Section 5: Per-subject figures ────────────────────────────────────
        for sub, evs in evokeds_by_subject.items():
            print(f'--- sub-{sub} ---')

            fig_erp = plot_evoked_comparison(
                evs, picks=PICKS_FOR_COMPARE,
                title=f'sub-{sub} task-{TASK} ref-{ref_label}',
            )
            plt.show()

            fig_topo_layout = None
            try:
                figs = mne.viz.plot_compare_evokeds(
                    evs, axes='topo', picks='eeg',
                    title=f'sub-{sub} task-{TASK} ref-{ref_label}',
                    invert_y=True, ylim=dict(eeg=YLIM),
                    ci=None,
                    show=False,
                )
                fig_topo_layout = figs[0] if isinstance(figs, list) else figs
                plt.show()
            except (RuntimeError, ValueError) as e:
                print(f'  topo layout skipped: {e}')

            if SAVE_PER_SUBJECT_FIGURES:
                save_kwargs = dict(bids_root=BIDS_ROOT, subject=sub, task=TASK, pipeline_name=PIPELINE_NAME)
                p = save_subject_figure(fig_erp, desc='erp-compare', **save_kwargs)
                print(f'  ERP figure: {p.name}')
                if fig_topo_layout is not None:
                    p = save_subject_figure(fig_topo_layout, desc='topo-layout', **save_kwargs)
                    print(f'  topo layout: {p.name}')


## 4. Trial count summary — all subjects × conditions × references

Reads the per-subject TSV files written by the main loop and assembles a single table. Rows = subject × reference pipeline; columns = stimulus condition (intensity × frequency).

In [ ]:
_rows = []
for _task, _ in TASKS.items():
    for _, _ref_label in REFERENCES:
        _pipeline = f'erp-pipeline-ref-{_ref_label}'
        _deriv = Path(BIDS_ROOT) / 'derivatives' / _pipeline
        for _sub in SUBJECTS_RESOLVED:
            _tsv = _deriv / f'sub-{_sub}' / 'eeg' / f'sub-{_sub}_task-{_task}_desc-trial-counts.tsv'
            if not _tsv.exists():
                continue
            _s = pd.read_csv(_tsv, sep='\t', index_col=0)['n_trials']
            for _cond, _n in _s.items():
                _rows.append({
                    'subject':   _sub,
                    'reference': _ref_label,
                    'task':      _task,
                    'condition': _cond,
                    'n_trials':  int(_n),
                })

# Total epochs presented per condition per subject (from BIDS events files — always 400)
_N_PRESENTED = 400

if not _rows:
    print('No trial-count TSV files found — run the main loop first.')
else:
    _df = pd.DataFrame(_rows)

    # Wide table: rows = (subject, reference), columns = condition
    _wide = (
        _df.pivot_table(index=['subject', 'reference'], columns='condition',
                        values='n_trials', aggfunc='sum')
        .astype('Int64')
    )
    _wide.columns.name = None

    from IPython.display import display
    print(f'Non-rejected epochs — {len(SUBJECTS_RESOLVED)} subject(s), '
          f'{len(REFERENCES)} reference(s), {_wide.shape[1]} conditions\n'
          f'(presented per condition per subject: {_N_PRESENTED})\n')
    display(_wide)

    # Per-subject row totals
    _wide_data = _wide.copy()
    _wide_data['TOTAL'] = _wide_data.sum(axis=1)
    print('\nRow totals (subject × reference):')
    display(_wide_data[['TOTAL']])

    # Per-subject min / max with percentage of presented epochs
    print('\nPer-subject min / max non-rejected epochs across conditions:')
    _summary_rows = []
    for _sub in SUBJECTS_RESOLVED:
        _sub_df = _df[_df['subject'] == _sub]
        if _sub_df.empty:
            continue
        for _ref in _sub_df['reference'].unique():
            _ref_df = _sub_df[_sub_df['reference'] == _ref]
            _min_n  = _ref_df['n_trials'].min()
            _max_n  = _ref_df['n_trials'].max()
            _min_cond = _ref_df.loc[_ref_df['n_trials'].idxmin(), 'condition']
            _max_cond = _ref_df.loc[_ref_df['n_trials'].idxmax(), 'condition']
            _summary_rows.append({
                'subject':   _sub,
                'reference': _ref,
                'min_n':     _min_n,
                'min_%':     f'{100 * _min_n / _N_PRESENTED:.0f}%',
                'min_cond':  _min_cond,
                'max_n':     _max_n,
                'max_%':     f'{100 * _max_n / _N_PRESENTED:.0f}%',
                'max_cond':  _max_cond,
            })

    _summary = pd.DataFrame(_summary_rows).set_index(['subject', 'reference'])
    display(_summary)

    print(f'\nGrand mean trials per condition: {_df["n_trials"].mean():.1f} / {_N_PRESENTED}  '
          f'({100 * _df["n_trials"].mean() / _N_PRESENTED:.1f}%  '
          f'min {_df["n_trials"].min()} ({100 * _df["n_trials"].min() / _N_PRESENTED:.0f}%), '
          f'max {_df["n_trials"].max()} ({100 * _df["n_trials"].max() / _N_PRESENTED:.0f}%))')
